<a href="https://colab.research.google.com/github/LauraAceros/Workshop3/blob/main/05_ejercicio_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Ejercicio: Asistente de Documentos con Gemini y Gradio

**Tiempo:** 40-45 minutos  
**Grupos:** 3-4 personas  

## Contexto

En los notebooks anteriores construimos chatbots que responden preguntas generales.
En este ejercicio daremos un paso más: construir un asistente que responda preguntas
sobre un documento específico — en este caso, el paper seminal de los Transformers:
**"Attention is All You Need"** (Vaswani et al., 2017).

Esta técnica — inyectar el contenido de un documento en el contexto del modelo —
es la base conceptual de **RAG (Retrieval-Augmented Generation)**, uno de los
patrones más usados en aplicaciones de IA en producción.

## Objetivo

Construir una aplicación Gradio donde el usuario pueda:
1. Cargar el PDF del paper
2. Hacer preguntas sobre su contenido
3. Obtener respuestas basadas **exclusivamente** en el documento

## Lo que van a aprender

- Extracción de texto desde PDFs con `pypdf`
- Inyección de contexto externo en el system prompt
- Limitaciones de este enfoque y por qué existe RAG
- Integración de `gr.File` en una interfaz Gradio

---

## Paso 0: Instalación y configuración

### 0.1 Instala las dependencias necesarias

Necesitarás tres bibliotecas nuevas además de las que ya conoces:
- `pypdf`: para extraer texto de archivos PDF
- `gradio`: para la interfaz web
- `google-genai`: para el modelo

```
pip install pypdf gradio google-genai python-dotenv
```

In [37]:
# Si estás en Colab/entorno limpio, descomenta esta línea:
!pip install -q pypdf gradio google-genai python-dotenv

print("Dependencias listas. Si falta alguna, ejecuta el pip de arriba.")

Dependencias listas. Si falta alguna, ejecuta el pip de arriba.


### 0.2 Descarga el paper

Descarga el PDF de ArXiv (acceso abierto):
```
https://arxiv.org/pdf/1706.03762
```

Guárdalo en la misma carpeta que este notebook con el nombre `attention_is_all_you_need.pdf`.

### 0.3 Configura tus credenciales

Crea un archivo `.env` con tu API key de Gemini:
```
GEMINI_API_KEY="Api_key_aqui"
```

In [38]:
# Descarga el PDF si aún no está en esta carpeta (mismo nombre que pide el ejercicio)
import urllib.request
from pathlib import Path

PDF_NAME = Path("attention_is_all_you_need.pdf")
PDF_URL = "https://arxiv.org/pdf/1706.03762"
if not PDF_NAME.exists():
    print("Descargando desde arXiv...")
    urllib.request.urlretrieve(PDF_URL, PDF_NAME)
    print("Listo:", PDF_NAME.resolve())
else:
    print("El PDF ya existe:", PDF_NAME.resolve())

El PDF ya existe: /content/attention_is_all_you_need.pdf


## Paso 1: Extracción de texto del PDF

Lo primero es leer el PDF y extraer su contenido como texto plano.
`pypdf` hace esto en pocas líneas.

**Instrucciones:**
1. Importa `PdfReader` desde `pypdf`
2. Crea una función `extract_text_from_pdf(pdf_path)` que:
   - Abra el PDF desde la ruta `pdf_path`
   - Itere sobre todas las páginas
   - Concatene el texto de cada página
   - Retorne el texto completo como string
3. Prueba la función con `attention_is_all_you_need.pdf`
4. Imprime los primeros 500 caracteres para verificar que funcionó

**Pista:** `PdfReader` tiene un atributo `pages` que es una lista.
Cada página tiene un método `extract_text()`.


In [39]:
from pypdf import PdfReader


def extract_text_from_pdf(pdf_path: str) -> str:
    """Extrae y concatena texto de todas las páginas de un PDF."""
    reader = PdfReader(pdf_path)
    text_parts = []
    for page in reader.pages:
        text_parts.append(page.extract_text() or "")
    return "\n".join(text_parts)


pdf_path = "attention_is_all_you_need.pdf"
document_text_preview = extract_text_from_pdf(pdf_path)
print("Primeros 500 caracteres:\n")
print(document_text_preview[:500])

Primeros 500 caracteres:

Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗ †
University of Toronto
aidan@cs.toronto.edu
Łukasz 


## Paso 2: Inicialización del cliente de Gemini

Igual que en los notebooks anteriores.

**Instrucciones:**
1. Importa las bibliotecas necesarias (`os`, `dotenv`, `google.genai`, `google.genai.types`)
2. Carga las variables de entorno
3. Inicializa el cliente de Gemini
4. Define la constante `MODELO = "gemini-2.5-flash-lite"`
5. Verifica que la API key esté disponible


In [40]:
import os
from dotenv import load_dotenv
from google import genai
from google.genai import types

load_dotenv("api_key.env")
api_key = os.getenv("GEMINI_API_KEY")

if not api_key:
    raise ValueError(
        "No se encontró GEMINI_API_KEY. Crea un archivo .env con GEMINI_API_KEY='tu_key'."
    )

cliente = genai.Client(api_key=api_key)
MODELO = "gemini-2.5-flash-lite"

print("Cliente Gemini inicializado correctamente.")
print("Modelo:", MODELO)

Cliente Gemini inicializado correctamente.
Modelo: gemini-2.5-flash-lite


## Paso 3: Función de chat con contexto del documento

Esta es la parte central del ejercicio. La idea es construir un system prompt
que incluya el texto completo del paper, instruyendo al modelo a responder
**solo** basándose en ese contenido.

**Instrucciones:**
1. Crea una función `build_system_prompt(document_text)` que reciba el texto
   del documento y retorne un system prompt que:
   - Defina el rol del asistente (experto en el paper)
   - Incluya el texto completo del documento
   - Instruya al modelo a responder **solo** con información del documento
   - Indique qué hacer si la respuesta no está en el documento

2. Crea una función `chat_con_documento(message, history, document_text)` que:
   - Construya el historial en formato Gemini (objetos `types.Content`)
   - Use `generate_content_stream` con el system prompt del documento
   - Retorne la respuesta con `yield` para streaming

**Pista:** Recuerda que en Gradio el historial llega como lista de dicts
con claves `role` y `content`. El rol del asistente en Gemini es `"model"`.


# Nota personal
Gradio guarda el historial como diccionarios simples (pares clave-valor):

`python{"role": "user", "content": "¿Cuántas capas tiene el encoder?"}`

Gemini en cambio espera objetos de su propia librería. Un objeto es como un diccionario pero más rígido — tiene una estructura fija definida por Google:


```
pythontypes.Content(
    role="user",
    parts=[types.Part(text="¿Cuántas capas tiene el encoder?")] <-LISTA: puede tener texto, imagen,etc.
)
```



In [41]:
def build_system_prompt(document_text: str) -> str:
    return f"""
Eres un asistente experto en el paper "Attention is All You Need".
Tu tarea es responder preguntas basándote EXCLUSIVAMENTE en el documento proporcionado.

Reglas obligatorias:
1) Usa solo información presente en el documento.
2) Si la respuesta no está explícita en el documento, responde:
   "No encuentro esa información en el documento proporcionado."
3) No inventes datos ni uses conocimiento externo.
4) Responde de forma clara y breve en español.

DOCUMENTO:
{document_text}
""".strip()


def _to_gemini_history(history):
    gemini_history = []
    for turn in history:
        if isinstance(turn, dict):
            role = "model" if turn["role"] == "assistant" else "user"
            content = turn.get("content", "") or ""
        else:
            # fallback formato tuplas
            user_msg, bot_msg = turn
            if user_msg:
                gemini_history.append(types.Content(role="user", parts=[types.Part(text=user_msg)]))
            if bot_msg:
                gemini_history.append(types.Content(role="model", parts=[types.Part(text=bot_msg)]))
            continue

        if content.strip():
            gemini_history.append(types.Content(role=role, parts=[types.Part(text=content)]))
    return gemini_history


def chat_con_documento(message, history, document_text):
    if not document_text or not document_text.strip():
        yield "No hay texto del documento cargado. Verifica el PDF."
        return

    system_prompt = build_system_prompt(document_text)
    contents = _to_gemini_history(history)
    contents.append(types.Content(role="user", parts=[types.Part(text=message)]))

    respuesta_parcial = ""
    #La palabra stream en el nombre significa que la respuesta no llega completa de un golpe — llega en pedazos. Cada pedazo se llama chunk.
    stream = cliente.models.generate_content_stream(
        model=MODELO,
        contents=contents, # el historial + el mensaje nuevo del usuario
        config=types.GenerateContentConfig(system_instruction=system_prompt), # el paper completo adentro
    )

    for chunk in stream:
        if chunk.text:
            respuesta_parcial += chunk.text
            yield respuesta_parcial

# NOTA PERSONAL

```
for chunk in stream:
        if chunk.text:
            respuesta_parcial += chunk.text
            yield respuesta_parcial
```


Cada vuelta del for es un pedazo nuevo de texto que llegó. Lo vas acumulando en respuesta_parcial y con yield se lo vas mostrando al usuario progresivamente.

## Paso 4: Interfaz Gradio

Ahora construimos la interfaz. Usaremos `gr.ChatInterface` con `additional_inputs`
para pasar el texto del documento extraído.

**Instrucciones:**
1. Extrae el texto del PDF usando la función del Paso 1
2. Imprime cuántas páginas tiene y cuántos caracteres extraíste
3. Construye la interfaz con `gr.ChatInterface`:
   - `fn`: tu función `chat_con_documento`
   - `title`: un título descriptivo
   - `description`: explica qué puede hacer el asistente
   - `additional_inputs`: un `gr.Textbox` visible=False que contenga el texto
     del documento (así Gradio lo pasa automáticamente a la función)
   - `examples`: al menos 3 preguntas relevantes sobre el paper

**Pista:** Para pasar el texto del documento sin mostrarlo en la interfaz:
```python
gr.Textbox(value=document_text, visible=False)
```

4. Lanza la interfaz con `demo.launch(server_name="0.0.0.0", server_port=8080)`


In [ ]:
import gradio as gr

pdf_path = "attention_is_all_you_need.pdf"
reader = PdfReader(pdf_path)
document_text = extract_text_from_pdf(pdf_path)

print(f"Páginas del PDF: {len(reader.pages)}")
print(f"Caracteres extraídos: {len(document_text):,}")

examples = [
    ["¿Cuál es la arquitectura principal propuesta en el paper?"],
    ["¿Qué son las multi-head attention layers?"],
    ["¿Cuántas capas tiene el encoder del modelo base?"],
]

# ChatInterface pasa adicional_inputs como argumentos extra a fn
# fn(message, history, document_text)
demo = gr.ChatInterface(
    fn=chat_con_documento,
    type="messages",
    title="Asistente del Paper: Attention is All You Need",
    description=(
        "Haz preguntas sobre el paper. El asistente debe responder solo con "
        "información del documento cargado."
    ),
    additional_inputs=[gr.Textbox(value=document_text, visible=False)],
    examples=examples,
)

# En local puedes cambiar share=True si necesitas URL pública
demo.launch(share=True, debug=True)

Páginas del PDF: 15
Caracteres extraídos: 39,629
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://000aaf1407012502ed.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## Paso 5: Prueba y reflexión

Una vez que la interfaz esté funcionando, prueba estas preguntas:

1. *"¿Cuál es la arquitectura principal propuesta en el paper?"*
2. *"¿Qué es el mecanismo de atención?"*
3. *"¿Cuántas capas tiene el encoder del modelo base?"*
4. *"¿Quiénes son los autores del paper?"*
5. *"¿Cuál es el resultado del modelo en la tarea WMT 2014 English-to-German?"*

Y esta pregunta trampa:
6. *"¿Qué es GPT-4?"*

Esta última pregunta **no está en el paper**. Observa cómo responde el modelo.
¿Usa su conocimiento general o respeta la instrucción de ceñirse al documento?


### Respuestas orientativas — Paso 5 (prueba y reflexión)

**Preguntas 1–5 (según el paper):** el asistente debería poder responder con base en el PDF: arquitectura **Transformer** (encoder–decoder), **Scaled Dot-Product Attention** y **Multi-Head Attention**, **6 capas** en el modelo base (encoder y decoder), autores **Vaswani et al.**, y resultados en **WMT 2014 English–German** (los **BLEU** exactos aparecen en las **tablas** del PDF; cítalos desde ahí).

**Pregunta 6 (trampa — “¿Qué es GPT-4?”):** GPT-4 **no aparece** en el paper de 2017. Lo esperable es que el modelo responda que **no está en el documento** (si respeta el system prompt). A veces el modelo **mezcla conocimiento general** con el documento; por eso conviene revisar la respuesta y, si hace falta, reforzar en el prompt que no cite nada fuera del texto.

## Paso 6 (Adicional): Mejora el sistema

Si terminaron antes de tiempo, implementen **una** de estas mejoras:

**A) Indicador de tokens**
Muestra cuántos tokens está usando el system prompt. Recuerda que Gemini 2.5 Flash
tiene un límite de 1,000,000 tokens — ¿qué porcentaje del límite usa este paper?

**B) Subida dinámica de PDF**
En lugar de cargar el PDF con código, agrega un `gr.File` en `additional_inputs`
para que el usuario pueda subir cualquier PDF desde la interfaz.
Tendrás que extraer el texto dentro de la función de chat cuando llegue el archivo.

**C) Citas del documento**
Modifica el system prompt para que el modelo siempre incluya una cita textual
del paper que respalde su respuesta.

In [ ]:
# Paso 6 (opción A): indicador aproximado de tokens del system prompt
# Ejecuta después de tener `document_text` (celda de Gradio) o usa el preview del Paso 1.

try:
    _txt = document_text
except NameError:
    _txt = extract_text_from_pdf("attention_is_all_you_need.pdf")

_sp = build_system_prompt(_txt)
# API oficial de conteo (si falla, estimación ~4 caracteres/token)
try:
    ct = cliente.models.count_tokens(model=MODELO, contents=_sp)
    n = getattr(ct, "total_tokens", None) or getattr(ct, "totalTokenCount", None) or 0
    print("Tokens (count_tokens API):", n)
except Exception as e:
    n = max(1, len(_sp) // 4)
    print("Tokens (estimación len//4):", n, "| API no disponible:", e)

LIMITE = 1_000_000
print(f"Porcentaje del límite ({LIMITE:,} tokens): {100 * n / LIMITE:.4f}%")

Tokens (count_tokens API): 10948
Porcentaje del límite (1,000,000 tokens): 1.0948%


## Reflexión final

Discutan en grupo:

1. **¿Cuál es la limitación principal de este enfoque?**
   Piensen en qué pasaría con un documento de 1000 páginas.

2. **¿Por qué existe RAG?**
   ¿Cómo resolvería RAG el problema que identificaron?

3. **¿Qué información podría "filtrarse" aunque el system prompt diga que no?**
   El modelo tiene conocimiento previo sobre el paper — ¿cómo verificarían
   que está respondiendo desde el documento y no desde su entrenamiento?

---

## Recursos

- [Documentación de pypdf](https://pypdf.readthedocs.io)
- [Documentación de Gradio](https://www.gradio.app/docs)
- [Documentación de Gemini API](https://ai.google.dev/gemini-api/docs)
- [Paper original en ArXiv](https://arxiv.org/abs/1706.03762)


### Respuestas orientativas — Reflexión final

1. **Limitación principal:** meter **todo el documento** en el prompt no escala. Con **1000 páginas** superarías el contexto del modelo, costaría mucho y sería lento; además el modelo puede “perder” detalle en medio de tanto texto.

2. **Por qué existe RAG:** en lugar de pegar el corpus entero, **se recuperan solo los fragmentos relevantes** (búsqueda por embeddings o BM25) y se mandan al modelo. Así caben documentos enormes, se reduce costo y se mejora la **fidelidad** a fuentes concretas.

3. **Filtraciones posibles:** el modelo puede **completar con memoria de entrenamiento** (p. ej. datos del paper que “recuerda” aunque no estén en el PDF que subiste). Para verificar: pedir **citas literales** del fragmento, comparar con el PDF, o usar **solo chunks recuperados** con trazabilidad (RAG + citas).